# 基础设施与部署（Infrastructure and Deployment）

前一节将训练好的模型接入实时行情，完成预测和模拟交易。本节进一步讨论：如果策略要使用真实资金持续运行，需要怎样的基础设施？

原文重点介绍可靠性、性能和安全性，以及云端部署与本地开发的区别。

## 1. 部署环境的基本要求

使用真实资金运行自动化交易策略，需要基础设施为策略提供稳定的运行环境。原书主要从以下三个方面提出要求。

### 1.1. 可靠性（Reliability）

交易系统应具备较高的可用性，例如 99.9% 或更高，并通过自动备份、磁盘冗余和网络连接冗余等措施提高可靠性。这些措施分别用于减少服务中断、降低数据丢失风险，以及避免单个设备或连接故障导致整个系统停止运行。

可用性可以理解为：在指定统计期间内，系统能够正常提供服务的时间占比。下面将原书提到的 99.9% 换算成停机时间，帮助理解这个比例。

### 1.2. 性能（Performance）

基础设施的配置应与处理的数据量和算法计算需求相适应，主要包括：

| 资源 | 需要满足的需求 |
|---|---|
| CPU | 提供足够的计算能力，支持数据处理和模型计算 |
| 内存（RAM） | 容纳运行中的程序、行情数据和计算结果 |
| 存储（SSD） | 满足数据、模型等文件的存储与读写需求 |
| 网络连接 | 具有足够的速度，支持行情接收和交易通信 |

配置应根据策略的实际负载选择。下面查看当前 Python 运行环境的基本资源，作为认识部署环境的补充示例。

In [1]:
import os
import platform
import shutil
from pathlib import Path

storage = shutil.disk_usage(Path.cwd())
gib = 1024 ** 3

print(f"操作系统：{platform.system()} {platform.release()}")
print(f"Python 版本：{platform.python_version()}")
print(f"逻辑 CPU 数：{os.cpu_count()}")
if platform.system() == 'Linux':
    memory = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES')
    print(f"系统报告的物理内存：{memory / gib:.2f} GiB")
print(f"当前目录所在文件系统总容量：{storage.total / gib:.2f} GiB")
print(f"当前目录所在文件系统可用容量：{storage.free / gib:.2f} GiB")

操作系统：Linux 7.0.0-38-generic
Python 版本：3.11.15
逻辑 CPU 数：16
系统报告的物理内存：31.13 GiB
当前目录所在文件系统总容量：467.35 GiB
当前目录所在文件系统可用容量：312.91 GiB


输出记录了执行代码时的操作系统、Python 版本、逻辑 CPU 数和存储容量；在 Linux 环境中，还会显示系统报告的物理内存。换一台机器或重新运行，可用存储等数值可能变化。

这些是资源信息，不能直接证明策略能够及时处理行情。逻辑 CPU 数不等于物理核心数，文件系统容量也不能说明底层磁盘是否为 SSD；在容器中，程序实际可用的资源还可能受到额外限制。本示例没有测量计算耗时、磁盘速度或网络性能。

### 1.3. 安全性（Security）

安全性既涉及操作系统和应用程序，也涉及物理设备。原书列出的措施包括：

- 使用强密码保护系统和应用程序的访问。
- 使用 SSL 加密保护网络通信，现代环境通常使用 TLS。
- 使用磁盘加密保护存储的数据。
- 保护硬件免受火灾、水损及未经授权的物理访问。

这些措施分别覆盖身份访问、传输数据、存储数据和物理环境。

## 2. 数据中心与云服务

为了满足上述可靠性、性能和安全性要求，原书建议租用专业数据中心或云服务商提供的基础设施。自行建设达到相应要求的物理环境，需要承担设备、机房、网络和维护等投入，通常只有规模较大的金融机构才有条件论证其经济合理性。

原书以 DigitalOcean 的 Droplet（云主机实例）为例：在开发和测试阶段，较小的实例即可用于起步。云实例可以较快创建和销毁，并按使用时间计费，便于进行实验。

**原书写作时**，其举例的最小 Droplet 价格为每月 5 美元，按小时计费，可在几分钟内创建、几秒内销毁。这些是书中的历史信息，不代表当前价格、计费规则或创建速度；原文脚注中的新用户优惠也属于历史推广信息。

具体创建方法见原书第 2 章 “Using Cloud Instances”。其中的 Bash 脚本可以按项目需要调整，例如修改要安装的 Python 软件包。下面补充一个简化的 Ubuntu 云主机环境配置示例。

租用云主机只是获得运行资源。策略程序及其运行配置仍需要部署和维护。

### 示例：配置云主机上的 Python 环境

假设已经创建 Ubuntu 云主机，并将 notebook、模型、配置文件和依赖清单 `requirements.txt` 上传至 `~/trading`。以下命令在云主机终端执行，属于补充示例，尚未实际运行。

In [7]:
# 安装 Python 虚拟环境工具
sudo apt update
sudo apt install -y python3-venv

# 为交易项目创建独立环境
cd ~/trading
python3 -m venv .venv
source .venv/bin/activate

# 按本地验证过的版本安装依赖，包括 JupyterLab
python -m pip install -r requirements.txt

# 启动 JupyterLab，仅监听云主机本地地址
jupyter lab --no-browser --ip=127.0.0.1 --port=8888


再在**本地电脑终端**建立 SSH 隧道，将用户名和地址替换为自己的云主机信息：

In [ ]:
# ssh -N -L 8888:127.0.0.1:8888 用户名@云主机地址

随后在本地浏览器访问 `http://127.0.0.1:8888`，使用启动日志中的 token 登录，即可打开 notebook 进行开发和模拟测试。

这一示例展示了云端环境的准备与访问方式。`requirements.txt` 应记录本地验证过的软件版本，尤其要保持 scikit-learn 版本一致，以便加载此前保存的模型。云主机的 Python 版本也应与项目环境兼容。

## 3. 本地开发与真实资金部署

本地台式机或笔记本适合开发和测试自动化交易策略。对于需要持续运行、使用真实资金的策略，原书不建议依赖普通本地电脑及其网络、电力环境。

原因在于，中断不仅意味着程序暂时停止，还可能改变交易结果：

| 故障 | 可能产生的影响 |
|---|---|
| 网络断开或短暂停电 | 程序停止运行，已有持仓可能无法按策略及时平仓或调整 |
| 实时行情接收中断 | 漏收 tick 数据，导致后续数据不完整 |
| 使用缺失或异常数据继续计算 | 特征和信号可能出错，进而产生非预期交易和持仓 |

因此，部署时需要关注的不只是代码能否启动，还包括它能否持续接收数据、处理行情并执行交易。前一节的本地模拟适合学习和验证在线流程，真实资金部署则需要进一步满足本节介绍的基础设施要求。

## 4. 本节要点

自动化交易策略的部署环境需要兼顾可靠性、性能和安全性。专业数据中心和云服务提供了获取这些基础资源的途径，本地电脑则适合开发和测试。

基础设施故障可能留下未按计划处理的持仓，也可能造成行情缺失，影响后续信号和交易。本节关注的是支撑策略持续运行的环境，具体云主机配置方法可衔接原书第 2 章学习。